In [3]:
pip install PyPDF2

  Using cached pypdf2-3.0.1-py3-none-any.whl.metadata (6.8 kB)
Using cached pypdf2-3.0.1-py3-none-any.whl (232 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from PyPDF2 import PdfReader


In [24]:
pdf_folder = r"C:\Users\Aqil Anhein\Desktop\TA\Peraturan Pemerintah\Turunan UU No 1 1970"

# Create output subfolder
output_folder = os.path.join(pdf_folder, "cleaned_txt")
os.makedirs(output_folder, exist_ok=True)

# Get all PDF files in the folder
pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith('.pdf')]

# Loop through each PDF file
for pdf_file in pdf_files:
    pdf_path = os.path.join(pdf_folder, pdf_file)
    txt_filename = os.path.splitext(pdf_file)[0] + ".txt"
    txt_path = os.path.join(output_folder, txt_filename)  # <-- save in new folder

    # Read the PDF
    with open(pdf_path, "rb") as f:
        reader = PdfReader(f)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""

    # Save to a .txt file in output folder
    with open(txt_path, "w", encoding="utf-8") as txt_file:
        txt_file.write(text)

    print(f"Extracted text from: {pdf_file} -> {txt_filename}")

Extracted text from: PP_No_19_Tahun_1973.pdf -> PP_No_19_Tahun_1973.txt
Extracted text from: PP_No_50_Tahun_2012bt.pdf -> PP_No_50_Tahun_2012bt.txt


CLEANING TEXT

In [4]:
import re

In [5]:
def remove_ellipsis_lines(text):
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        # Remove line if it contains '...' or '…' or similar ellipsis patterns
        if re.search(r'\.\.\.|…', line):
            continue
        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


def remove_footer_clutter(text):
    # Step 1: Merge lines to detect cross-line footer phrases
    text = re.sub("PRESIDEN\nREPUBLIK INDONESIA", "", text)
    # Step 2: Forcefully remove case-sensitive footer parts
    text = re.sub("PRESIDEN", "", text)
    text = re.sub("REPUBLIK INDONESIA", "", text)
    text = re.sub(
        r'[\w.,;:\)\(]*PRESIDEN\s*[\n ]*REPUBLIK INDONESIA',
        '',
        text
    )
    text = re.sub(
        r'\.?\s*PRESIDEN\s*\n\s*REPUBLIK INDONESIA',
        '',
        text
    )
    # Step 3: Remove page numbers like -1-, -23- (exact match on a line)
    #text = re.sub(r'(?m)^-\d+-$', '', text)
    text = re.sub(r'(?m)^\s*-\s*\d+\s*-\s*$', '', text)
    # Step 4: Remove URLs
    text = re.sub(r'^.*(https?://\S+|www\.\S+).*\n?', '', text, flags=re.MULTILINE)
    return text


def merge_bab_titles(text):
    lines = text.splitlines()
    merged_lines = []
    i = 0
    while i < len(lines):
        line = lines[i].strip()

        # Match lines that are only "BAB IX", "BAB V", etc. — nothing else
        if re.fullmatch(r'BAB\s+[IVXLCDM]+', line, flags=re.IGNORECASE):
            # Next line should be the title
            if i + 1 < len(lines):
                title_line = lines[i + 1].strip()
                # Merge and force to uppercase
                merged_lines.append(f"{line.upper()} : {title_line}")
                i += 2  # Skip next line
                continue

        # Otherwise, keep the line as-is
        merged_lines.append(lines[i])
        i += 1
    return "\n".join(merged_lines)


def uppercase_pasal_headers(text, max_length=12):
    lines = text.splitlines()
    updated_lines = []

    for line in lines:
        stripped = line.strip()

        # Check if line starts with "Pasal" and is shorter than the threshold
        if re.match(r'^Pasal\b.*$', stripped, flags=re.IGNORECASE) and len(stripped) <= max_length:
            updated_lines.append(stripped.upper())
        else:
            updated_lines.append(line)

    return "\n".join(updated_lines)
    

def remove_after_first_signature_block(text):
    # Signature start patterns
    pattern = r'(?:Ditetapkan|Disahkan|Diundangkan)\s+di\s+Jakarta|Agar setiap orang mengetahuinya|di\s+Jakarta'

    match = re.search(pattern, text, flags=re.IGNORECASE)

    if not match:
        return text  # no signature block found

    return text[:match.start()].strip()

In [6]:
# Compile patterns once
MAIN_PATTERN = re.compile(
    r'\b(M|IVI|IM|I\W?I)?[EI1]{1,2}M?U[VvT7]U[S5]?[ \n]*K[AI1]N[:：．. ]?', 
    flags=re.IGNORECASE
)
FALLBACK_PATTERN = re.compile(
    r'[\.\-–—…]*[ \n]*UTUSKAN[:：．. ]?', 
    flags=re.IGNORECASE
)
SPACED_LETTERS_PATTERN = re.compile(
    r'(?:(?:\b|\s)[Mm][\s]*)'
    r'(?:[Ee][\s]*)'
    r'(?:[Mm][\s]*)'
    r'(?:[Uu][\s]*)'
    r'(?:[Tt][\s]*)'
    r'(?:[Uu][\s]*)'
    r'(?:[Ss][\s]*)'
    r'(?:[Kk][\s]*)'
    r'(?:[Aa][\s]*)'
    r'(?:[Nn])'
    r'(?:[\s:：．.]+)?',
    flags=re.IGNORECASE
)

def normalize_and_extract_memutuskan(text, filename="<unknown>"):
    if not isinstance(text, str):
        print(f"[WARN] Invalid text input in file: {filename}")
        return ""

    # Step 1: Try main pattern
    text, count = MAIN_PATTERN.subn('MEMUTUSKAN', text, count=1)

    # Step 2: Fallback to UTUSKAN variants
    if count == 0:
        text, fallback_count = FALLBACK_PATTERN.subn('MEMUTUSKAN', text, count=1)
        count += fallback_count

    # Step 3: Fallback to spaced letters
    if count == 0:
        text, spaced_count = SPACED_LETTERS_PATTERN.subn('MEMUTUSKAN', text, count=1)
        count += spaced_count

    # Final extraction
    if count == 0:
        print(f"[WARN] MEMUTUSKAN not found in file: {filename}")
        return text

    split = text.split('MEMUTUSKAN', 1)
    if len(split) > 1:
        return 'MEMUTUSKAN' + split[1].strip()
    else:
        print(f"[WARN] Failed to split after MEMUTUSKAN in file: {filename}")
        return text

In [7]:
def force_missing_pasal_on_newline(text):
    header_pattern = re.compile(r'^\s*Pasal\s+((\d\s*){1,3})\b', re.IGNORECASE)
    lines = text.splitlines()
    pasal_numbers = []

    # Step 1: Collect all PASAL headers
    for line in lines:
        match = header_pattern.match(line.strip())
        if match:
            raw_number = match.group(1)
            pasal_num = int(re.sub(r'\s+', '', raw_number))
            pasal_numbers.append(pasal_num)

    # Step 2: Detect missing pasals
    missing_pasal_numbers = []
    for i in range(1, len(pasal_numbers)):
        current = pasal_numbers[i]
        previous = pasal_numbers[i - 1]
        if current > previous + 1:
            missing_pasal_numbers.extend(range(previous + 1, current))

    if missing_pasal_numbers:
        print("Missing PASALs detected:", missing_pasal_numbers)

    # Step 3: For each missing, insert "\nPASAL <n>\n" even if glued
    for missing in missing_pasal_numbers:
        # Match even when glued: e.g., "abcPasal 8"
        pattern = re.compile(rf'(?i)(?<!\n)(?<!^)(?P<before>.{{0,20}}?)pasal\s+{missing}\b')
        text = pattern.sub(r'\g<before>\nPASAL ' + str(missing) + r'\n', text)

    return text

In [9]:
def uppercase_pasal_headers(text, max_length=12):
    lines = text.splitlines()
    output = []

    pasal_header_pattern = re.compile(r'^Pasal\s+((\d\s*){1,3})\b.*$', re.IGNORECASE)

    last_pasal_num = None
    last_pasal_index = None

    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.strip()

        header_match = pasal_header_pattern.match(stripped)

        if header_match:
            # Count total characters without spaces
            char_count = len(stripped.replace(" ", ""))
            if char_count <= max_length:
                # Normalize pasal number
                raw_number = header_match.group(1)
                pasal_num = int(re.sub(r'\s+', '', raw_number))

                # 🔍 If a pasal is skipped, backtrack and insert missing one(s)
                if last_pasal_num is not None and pasal_num > last_pasal_num + 1:
                    block_start = last_pasal_index + 1
                    block_end = len(output)
                    block = "\n".join(output[block_start:block_end])

                    for missing in range(last_pasal_num + 1, pasal_num):
                        # Match glued or loose "pasal X"
                        pattern = re.compile(rf'(?i)(?P<before>.{{0,20}}?)pasal\s+{missing}\b')
                        match = pattern.search(block)
                        if match:
                            print(f"✅ Inserting PASAL {missing} (missing between PASAL {last_pasal_num} and PASAL {pasal_num})")
                            insert_pos = match.start()
                            block = (
                                block[:insert_pos] +
                                f'\nPASAL {missing}\n' +
                                block[insert_pos:]
                            )

                    # Replace that chunk in output
                    output = output[:block_start] + block.splitlines()

                # Output corrected header
                fixed_line = f'PASAL {pasal_num}'
                output.append(fixed_line)

                last_pasal_num = pasal_num
                last_pasal_index = len(output) - 1
                i += 1
                continue  # Skip raw line

        # Not a header, just append
        output.append(line)
        i += 1

    return "\n".join(output)

In [26]:
def process_txt_files(folder_path):
    # Create output folder
    output_folder = os.path.join(folder_path, "final_cleaned")
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt") and not filename.endswith("_cleaned.txt"):
            file_path = os.path.join(folder_path, filename)
            
            # Read original file
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
            
            # Process content
            cleaned_content = normalize_and_extract_memutuskan(content,file_path)
            cleaned_content = remove_ellipsis_lines(cleaned_content)
            cleaned_content = remove_footer_clutter(cleaned_content)
            cleaned_content = remove_after_first_signature_block(cleaned_content)
            cleaned_content = merge_bab_titles(cleaned_content)
            #cleaned_content = force_missing_pasal_on_newline(cleaned_content)
            cleaned_content = uppercase_pasal_headers(cleaned_content)
            
            # Save to new file in output folder
            new_filename = filename.replace('.txt', '_cleaned.txt')
            new_path = os.path.join(output_folder, new_filename)
            
            with open(new_path, 'w', encoding='utf-8') as f:
                f.write(cleaned_content)
            
            print(f"Processed: {filename} -> final_cleaned/{new_filename}")

# Example usage
folder_path = r"C:\Users\Aqil Anhein\Desktop\TA\Peraturan Pemerintah\cleaned_txt"
process_txt_files(folder_path)

Processed: PP_No_29-Tahun_2021.txt -> final_cleaned/PP_No_29-Tahun_2021_cleaned.txt
